In [11]:
pip install selenium webdriver-manager beautifulsoup4 pandas lxml tqdm  #selenium not used here

Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [13]:
# -------------------------------
# 1. Disease → URL mapping
# -------------------------------
disease_url_map = {
    "Influenza": "https://medlineplus.gov/flu.html",
    "COVID-19": "https://medlineplus.gov/covid19.html",
    "Tuberculosis": "https://medlineplus.gov/tuberculosis.html",
    "Malaria": "https://medlineplus.gov/malaria.html",
    "Dengue fever": "https://medlineplus.gov/dengue.html",
    "Chickenpox": "https://medlineplus.gov/chickenpox.html",
    "Measles": "https://medlineplus.gov/measles.html",
    "Typhoid fever": "https://medlineplus.gov/typhoidfever.html",
    "Hepatitis B": "https://medlineplus.gov/hepatitisb.html",
    "Hepatitis C": "https://medlineplus.gov/hepatitisc.html",
    "Asthma": "https://medlineplus.gov/asthma.html",
    "Pneumonia": "https://medlineplus.gov/pneumonia.html",
    "Hypertension": "https://medlineplus.gov/highbloodpressure.html",
    "Diabetes": "https://medlineplus.gov/diabetes.html",
    "Migraine": "https://medlineplus.gov/migraine.html",
    "Depression": "https://medlineplus.gov/depression.html",
}

In [14]:
# -------------------------------
# 2. Extract ONLY symptom section
# -------------------------------
def extract_symptoms(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    symptoms = []

    for tag in soup.find_all(["h2", "h3"]):
        if "symptom" in tag.text.lower():

            next_node = tag.find_next_sibling()

            while next_node and next_node.name in ["ul", "p"]:
                
                if next_node.name == "ul":
                    for li in next_node.find_all("li"):
                        symptoms.append(li.text.strip())

                elif next_node.name == "p":
                    symptoms.append(next_node.text.strip())

                next_node = next_node.find_next_sibling()

    return symptoms

In [15]:
 #3. Clean symptoms
# -------------------------------
def clean_symptoms(symptoms):
    cleaned = []

    for s in symptoms:
        s = s.lower()
        s = re.sub(r"\(.*?\)", "", s)

        parts = re.split(r",|and", s)

        for p in parts:
            p = p.strip()

            if len(p) > 3 and not p.startswith(("your", "the", "a ")):
                cleaned.append(p)

    return list(set(cleaned))

In [16]:
# 4. Normalize
# -------------------------------
def normalize(sym):
    sym = sym.replace(" ", "_")

    junk = ["you", "your", "doctor", "will", "may"]
    if any(j in sym for j in junk):
        return None

    return sym

In [17]:
# 5. Scrape all diseases
# -------------------------------
data = []

for disease, url in disease_url_map.items():
    print("Scraping:", disease)

    raw = extract_symptoms(url)
    cleaned = clean_symptoms(raw)

    normalized = [normalize(s) for s in cleaned]
    normalized = [s for s in normalized if s is not None]

    data.append({
        "disease": disease,
        "symptoms": list(set(normalized))
    })

    time.sleep(1)

Scraping: Influenza
Scraping: COVID-19
Scraping: Tuberculosis
Scraping: Malaria
Scraping: Dengue fever
Scraping: Chickenpox
Scraping: Measles
Scraping: Typhoid fever
Scraping: Hepatitis B
Scraping: Hepatitis C
Scraping: Asthma
Scraping: Pneumonia
Scraping: Hypertension
Scraping: Diabetes
Scraping: Migraine
Scraping: Depression


In [18]:
# 6. Convert to ML dataset
# -------------------------------
all_symptoms = sorted(set(sum([d["symptoms"] for d in data], [])))

rows = []

for d in data:
    row = {sym: 0 for sym in all_symptoms}
    for s in d["symptoms"]:
        row[s] = 1
    row["disease"] = d["disease"]
    rows.append(row)

df = pd.DataFrame(rows)

# -------------------------------
# 7. Save
# -------------------------------
df.to_csv("clean_disease_dataset.csv", index=False)

print("✅ Clean dataset ready!")

✅ Clean dataset ready!


In [23]:
print(df.columns)

Index(['/or_vomiting', '28_weeks_of_pregnancy.', 'abdominal_pain', 'aches',
       'anxious', 'are_pregnant', 'bleeding_from_the_nose_or_gums',
       'blurry_vision', 'called_jaundice', 'can't_spread_tb_to_others',
       ...
       'vomiting_blood', 'watch_for_signs', 'weak', 'weakness_or_fatigue',
       'wheezing', 'with_no_energy', 'with_tb_disease', 'with_type_2_diabetes',
       'yellowish_eyes', 'disease'],
      dtype='object', length=122)
